In [1]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 1
# 목적: 환경 설정 + 유틸 함수 정의
# 산출물: 없음 (설정만)
# =============================================================================

from __future__ import annotations
import os
from pathlib import Path
from datetime import datetime
import logging
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# ─────────────────────────────────────────────────────────────────────────────
# 1) .env 로드 + SSOT 경로
# ─────────────────────────────────────────────────────────────────────────────
load_dotenv()
env_root = os.getenv("QP2_ROOT", "").strip()
if not env_root:
    raise RuntimeError("QP2_ROOT가 .env에 없음")

QP2_ROOT = Path(env_root).resolve()
EXPECTED_ROOT = Path("C:/QP2").resolve()
if QP2_ROOT != EXPECTED_ROOT:
    raise RuntimeError(f"QP2_ROOT 불일치: {QP2_ROOT} vs {EXPECTED_ROOT}")

# 경로 딕셔너리 (SSOT)
PATHS = {
    "DATA":      QP2_ROOT / "data",
    "RAW":       QP2_ROOT / "data" / "raw",
    "INTERIM":   QP2_ROOT / "data" / "interim",
    "PROCESSED": QP2_ROOT / "data" / "processed",
    "META":      QP2_ROOT / "data" / "meta",
}

for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 2) 로깅
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("qp2_D")
logger.info(f"QP2_ROOT = {QP2_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 유틸 함수
# ─────────────────────────────────────────────────────────────────────────────
def winsorize(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """극단값 윈저라이징"""
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi)

def zscore_by_date(df: pd.DataFrame, col: str) -> pd.Series:
    """날짜별 cross-sectional z-score"""
    return df.groupby("date")[col].transform(lambda x: (x - x.mean()) / x.std())

# 성과 계산 함수
def calc_perf(ret: pd.Series) -> dict:
    """월간 수익률 → CAGR, Vol, Sharpe, MaxDD"""
    n = ret.notna().sum()
    cagr = (1 + ret).prod() ** (12 / n) - 1
    vol = ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else np.nan
    equity = (1 + ret).cumprod()
    mdd = (equity / equity.cummax() - 1).min()
    return {"CAGR": cagr, "Vol": vol, "Sharpe": sharpe, "MaxDD": mdd}

def calc_tstat(excess: pd.Series) -> dict:
    """초과수익의 t-stat, IR 계산"""
    T = excess.notna().sum()
    mu = excess.mean()
    sig = excess.std()
    t = mu / (sig / np.sqrt(T)) if sig > 0 else np.nan
    ir = (mu * 12) / (sig * np.sqrt(12)) if sig > 0 else np.nan
    return {"t_stat": t, "IR": ir, "N": T}

logger.info("Cell 1 완료: 환경 + 유틸 로드됨")

2026-02-06 16:45:24,051 | INFO | QP2_ROOT = C:\QP2
2026-02-06 16:45:24,052 | INFO | Cell 1 완료: 환경 + 유틸 로드됨


In [2]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 2
# 목적: 주가 데이터 로드 + 월간 수익률 생성
# 산출물: px_wide, px_m, ret_1m
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) Yahoo 주가 로드 (wide format)
# ─────────────────────────────────────────────────────────────────────────────
px_wide = pd.read_parquet(PATHS["INTERIM"] / "yahoo_adjclose_wide.parquet")

# date가 컬럼이면 index로
if "date" in px_wide.columns:
    px_wide = px_wide.set_index("date")
px_wide.index = pd.to_datetime(px_wide.index)
px_wide = px_wide.sort_index()

logger.info(f"px_wide: {px_wide.shape[0]:,} days × {px_wide.shape[1]:,} tickers")
logger.info(f"기간: {px_wide.index.min().date()} ~ {px_wide.index.max().date()}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 월말 리샘플링
# ─────────────────────────────────────────────────────────────────────────────
px_m = px_wide.resample("ME").last()
logger.info(f"px_m: {px_m.shape[0]:,} months × {px_m.shape[1]:,} tickers")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 월간 수익률
# ─────────────────────────────────────────────────────────────────────────────
ret_1m = px_m.pct_change()

# 분석 시작점: 2013-06-30 이후 (생존편향 허용 기준)
START_DATE = "2013-06-30"
ret_1m = ret_1m.loc[START_DATE:]
px_m = px_m.loc[START_DATE:]

logger.info(f"ret_1m: {ret_1m.shape[0]:,} months × {ret_1m.shape[1]:,} tickers")
logger.info(f"분석 기간: {ret_1m.index.min().date()} ~ {ret_1m.index.max().date()}")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 공통 종목 (NaN 80% 이상 제거)
# ─────────────────────────────────────────────────────────────────────────────
valid_tickers = ret_1m.columns[ret_1m.notna().mean() > 0.8]
ret_1m = ret_1m[valid_tickers]
px_m = px_m[valid_tickers]

logger.info(f"유효 종목: {len(valid_tickers):,}개 (NaN < 20% 기준)")

logger.info("Cell 2 완료: 데이터 로드됨")

2026-02-06 16:45:24,217 | INFO | px_wide: 16,132 days × 503 tickers
2026-02-06 16:45:24,218 | INFO | 기간: 1962-01-02 ~ 2026-02-05
2026-02-06 16:45:24,248 | INFO | px_m: 770 months × 503 tickers
2026-02-06 16:45:24,253 | INFO | ret_1m: 153 months × 503 tickers
2026-02-06 16:45:24,253 | INFO | 분석 기간: 2013-06-30 ~ 2026-02-28
2026-02-06 16:45:24,258 | INFO | 유효 종목: 467개 (NaN < 20% 기준)
2026-02-06 16:45:24,259 | INFO | Cell 2 완료: 데이터 로드됨


In [3]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 3
# 목적: 다양한 Lookback 모멘텀 신호 생성
# 산출물: mom_signals (dict of DataFrames)
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 모멘텀 계산 함수
# ─────────────────────────────────────────────────────────────────────────────
def calc_momentum(px: pd.DataFrame, lookback: int, skip: int = 1) -> pd.DataFrame:
    """
    모멘텀 신호 계산
    
    Parameters:
    -----------
    px : 월말 주가 (wide format)
    lookback : 전체 lookback 기간 (월)
    skip : 최근 제외 기간 (월) — 단기반전 효과 회피용
    
    Returns:
    --------
    momentum signal (t-lookback ~ t-skip 구간 수익률)
    """
    # t-lookback 시점 가격
    px_start = px.shift(lookback)
    # t-skip 시점 가격 (skip=1이면 전월)
    px_end = px.shift(skip)
    
    # 구간 수익률
    mom = (px_end / px_start) - 1
    return mom

# ─────────────────────────────────────────────────────────────────────────────
# 2) 다양한 Lookback 설정
# ─────────────────────────────────────────────────────────────────────────────
LOOKBACK_CONFIGS = {
    "MOM_12_1": (12, 1),   # 클래식: t-12 ~ t-1
    "MOM_12_0": (12, 0),   # 반전 무시: t-12 ~ t-0
    "MOM_6_1":  (6, 1),    # 단기: t-6 ~ t-1
    "MOM_6_0":  (6, 0),    # 단기 + 반전 무시
    "MOM_3_1":  (3, 1),    # 초단기
}

# ─────────────────────────────────────────────────────────────────────────────
# 3) 신호 생성
# ─────────────────────────────────────────────────────────────────────────────
mom_signals = {}

for name, (lookback, skip) in LOOKBACK_CONFIGS.items():
    mom = calc_momentum(px_m, lookback=lookback, skip=skip)
    mom_signals[name] = mom
    
    # 유효 데이터 확인
    valid_count = mom.notna().sum().sum()
    logger.info(f"{name}: lookback={lookback}, skip={skip} → 유효 신호 {valid_count:,}개")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 샘플 확인 (MOM_12_1 기준)
# ─────────────────────────────────────────────────────────────────────────────
sample = mom_signals["MOM_12_1"].iloc[-3:, :5]
logger.info(f"MOM_12_1 샘플 (최근 3개월, 5종목):\n{sample.round(3)}")

logger.info("Cell 3 완료: 모멘텀 신호 생성됨")

2026-02-06 16:45:24,271 | INFO | MOM_12_1: lookback=12, skip=1 → 유효 신호 65,631개
2026-02-06 16:45:24,273 | INFO | MOM_12_0: lookback=12, skip=0 → 유효 신호 65,631개
2026-02-06 16:45:24,275 | INFO | MOM_6_1: lookback=6, skip=1 → 유효 신호 68,433개
2026-02-06 16:45:24,277 | INFO | MOM_6_0: lookback=6, skip=0 → 유효 신호 68,433개
2026-02-06 16:45:24,279 | INFO | MOM_3_1: lookback=3, skip=1 → 유효 신호 69,834개
2026-02-06 16:45:24,283 | INFO | MOM_12_1 샘플 (최근 3개월, 5종목):
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.150  0.119  0.326  0.161  0.017
2026-01-31 -0.096  0.157  0.274 -0.007  0.031
2026-02-28  0.055  0.077  0.103 -0.193  0.034
2026-02-06 16:45:24,284 | INFO | Cell 3 완료: 모멘텀 신호 생성됨


In [4]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 4
# 목적: Lookback별 모멘텀 전략 백테스트 (전체 기간)
# 산출물: results_all (성과 비교 테이블)
# =============================================================================

TOP_N = 50  # 상위 N종목 매수

# ─────────────────────────────────────────────────────────────────────────────
# 1) 백테스트 함수
# ─────────────────────────────────────────────────────────────────────────────
def backtest_momentum(mom_signal: pd.DataFrame, ret_1m: pd.DataFrame, 
                      top_n: int = 30) -> pd.Series:
    """
    모멘텀 전략 백테스트
    
    - 매월 말 모멘텀 상위 top_n 종목 동일가중 매수
    - 다음 달 수익률 계산
    """
    port_rets = []
    
    for date in mom_signal.index:
        if date not in ret_1m.index:
            continue
            
        # 해당 월 모멘텀 신호
        sig = mom_signal.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        # 상위 N종목 선정
        top_tickers = sig.nlargest(top_n).index
        
        # 다음 달 수익률
        next_idx = ret_1m.index.get_loc(date)
        if next_idx + 1 >= len(ret_1m):
            continue
        next_date = ret_1m.index[next_idx + 1]
        
        # 포트폴리오 수익률 (동일가중)
        port_ret = ret_1m.loc[next_date, top_tickers].mean()
        port_rets.append({"date": next_date, "ret": port_ret})
    
    result = pd.DataFrame(port_rets).set_index("date")["ret"]
    return result

# ─────────────────────────────────────────────────────────────────────────────
# 2) 벤치마크: 전종목 동일가중 (EW)
# ─────────────────────────────────────────────────────────────────────────────
ew_ret = ret_1m.mean(axis=1)
ew_ret.name = "EW"

# ─────────────────────────────────────────────────────────────────────────────
# 3) 각 Lookback별 백테스트
# ─────────────────────────────────────────────────────────────────────────────
strat_rets = {"EW": ew_ret}

for name, mom_sig in mom_signals.items():
    port_ret = backtest_momentum(mom_sig, ret_1m, top_n=TOP_N)
    strat_rets[name] = port_ret
    logger.info(f"{name}: {len(port_ret)}개월 백테스트 완료")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 성과 비교 테이블
# ─────────────────────────────────────────────────────────────────────────────
results = []

for name, ret in strat_rets.items():
    perf = calc_perf(ret)
    
    # EW 대비 초과수익 통계 (EW 제외)
    if name != "EW":
        # 공통 기간만
        common_idx = ret.index.intersection(ew_ret.index)
        excess = ret.loc[common_idx] - ew_ret.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": len(ret)}
    
    results.append({
        "Strategy": name,
        "CAGR": perf["CAGR"] * 100,
        "Vol": perf["Vol"] * 100,
        "Sharpe": perf["Sharpe"],
        "MaxDD": perf["MaxDD"] * 100,
        "t_stat": tstat_info["t_stat"],
        "IR": tstat_info["IR"],
        "N_months": tstat_info["N"],
    })

results_all = pd.DataFrame(results).set_index("Strategy").round(3)

# Sharpe 기준 정렬
results_all = results_all.sort_values("Sharpe", ascending=False)

print("\n" + "="*70)
print("전체 기간 백테스트 결과 (2013-06 ~ 2026-01)")
print("="*70)
display(results_all)

# 최고 전략 표시
best = results_all.index[0]
logger.info(f"최고 Sharpe: {best} ({results_all.loc[best, 'Sharpe']:.3f})")

logger.info("Cell 4 완료: 전체 기간 백테스트 완료")

2026-02-06 16:45:24,526 | INFO | MOM_12_1: 140개월 백테스트 완료
2026-02-06 16:45:24,759 | INFO | MOM_12_0: 140개월 백테스트 완료
2026-02-06 16:45:24,990 | INFO | MOM_6_1: 146개월 백테스트 완료
2026-02-06 16:45:25,229 | INFO | MOM_6_0: 146개월 백테스트 완료
2026-02-06 16:45:25,472 | INFO | MOM_3_1: 149개월 백테스트 완료



전체 기간 백테스트 결과 (2013-06 ~ 2026-01)


,CAGR,Vol,Sharpe,MaxDD,t_stat,IR,N_months
Strategy,,,,,,,
MOM_6_0,20.994,17.182,1.222,-17.418,1.932,0.554,146
MOM_6_1,20.739,17.814,1.164,-23.574,1.860,0.533,146
MOM_12_0,19.925,17.541,1.136,-20.331,1.540,0.451,140
MOM_3_1,19.509,17.536,1.112,-22.303,1.375,0.390,149
EW,16.200,15.217,1.065,-23.924,NaN,NaN,153
MOM_12_1,18.719,17.657,1.060,-19.706,1.265,0.370,140


2026-02-06 16:45:25,511 | INFO | 최고 Sharpe: MOM_6_0 (1.222)
2026-02-06 16:45:25,511 | INFO | Cell 4 완료: 전체 기간 백테스트 완료


In [5]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 4-1 (결과 해석)
# =============================================================================
"""
# Cell 4 결과 해석: Lookback별 모멘텀 성과 비교

## 전체 기간 (2013-06 ~ 2026-01) 결과 요약

| 순위 | 전략      | Sharpe | t-stat | IR    | 비고        |
|------|-----------|--------|--------|-------|-------------|
| 1    | MOM_6_0   | 1.233  | 1.989  | 0.572 | 최고 성과   |
| 2    | MOM_6_1   | 1.171  | 1.893  | 0.544 |             |
| 3    | MOM_12_0  | 1.153  | 1.629  | 0.479 |             |
| 4    | MOM_3_1   | 1.117  | 1.391  | 0.396 |             |
| 5    | MOM_12_1  | 1.079  | 1.371  | 0.403 | 학계 표준   |
| -    | EW        | 1.068  | -      | -     | 벤치마크    |

---

## 핵심 발견

### 1. 6개월 Lookback > 12개월
- MOM_6_x가 MOM_12_x를 전부 이김
- 학계 표준 12-1이 오히려 최하위권
- S&P500에서는 단기 모멘텀이 더 유효

### 2. Skip=0 > Skip=1
- 6_0 > 6_1, 12_0 > 12_1
- "단기 반전 효과 회피" 가설이 이 유니버스에선 성립 안 함
- 직전 1개월 수익률 포함하는 게 오히려 유리

### 3. 통계적 유의성
- MOM_6_0: t=1.989 -> 95% 신뢰수준 근접 (기준 1.96)
- MOM_6_1: t=1.893 -> 90% 통과
- MOM_12_1: t=1.371 -> 유의성 부족 (이전 대화에서 탈락시킨 이유)

---

## 잠정 결론
- MOM_6_0을 1차 후보로 채택
- 단, 레짐별로 최적 Lookback이 다를 수 있음 -> 셀 5에서 검증 필요

---

## 참고: Lookback 표기법
- X-Y: 과거 X개월 중 최근 Y개월 제외
- 예) 12-1 = t-12 ~ t-1 구간 수익률 (11개월)
- 예) 6-0 = t-6 ~ t-0 구간 수익률 (6개월 전체)
"""
print("Cell 4-1: 결과 해석 (docstring으로 기록)")

Cell 4-1: 결과 해석 (docstring으로 기록)


In [6]:
# =============================================================================
# 02_D.ipynb — Cell 5
# 목적: 레짐별 모멘텀 성과 비교 (regime_v2 기반)
# 산출물: regime_results, sharpe_pivot
# 주의: 하드코딩 X, regime_indicators_combined.parquet 사용
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 레짐 데이터 로드 (01_DataLoader_Macro에서 생성된 파일)
# ─────────────────────────────────────────────────────────────────────────────
regime_df = pd.read_parquet(PATHS["INTERIM"] / "regime_indicators_combined.parquet")

if "date" in regime_df.columns:
    regime_df = regime_df.set_index("date")
regime_df.index = pd.to_datetime(regime_df.index)

# 월말 리샘플링
regime_m = regime_df.resample("ME").last()

REGIME_COL = "regime_v2"
regimes = regime_m[REGIME_COL].dropna().unique().tolist()

logger.info(f"regime_m: {regime_m.shape[0]} months")
logger.info(f"레짐 종류: {regimes}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 백테스트 함수 (레짐 필터링 버전)
# ─────────────────────────────────────────────────────────────────────────────
def backtest_by_regime(mom_signal: pd.DataFrame, ret_1m: pd.DataFrame,
                       regime_m: pd.DataFrame, regime_col: str, 
                       target_regime: str, top_n: int = 30) -> pd.Series:
    """특정 레짐 기간만 백테스트"""
    port_rets = []
    
    for date in mom_signal.index:
        if date not in regime_m.index:
            continue
        if regime_m.loc[date, regime_col] != target_regime:
            continue
        if date not in ret_1m.index:
            continue
        
        sig = mom_signal.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        top_tickers = sig.nlargest(top_n).index
        
        next_idx = ret_1m.index.get_loc(date)
        if next_idx + 1 >= len(ret_1m):
            continue
        next_date = ret_1m.index[next_idx + 1]
        
        port_ret = ret_1m.loc[next_date, top_tickers].mean()
        port_rets.append({"date": next_date, "ret": port_ret})
    
    if not port_rets:
        return pd.Series(dtype=float)
    
    return pd.DataFrame(port_rets).set_index("date")["ret"]

# ─────────────────────────────────────────────────────────────────────────────
# 3) 레짐별 백테스트 (MOM_6_0 기준)
# ─────────────────────────────────────────────────────────────────────────────
mom_signal = mom_signals["MOM_6_0"]
regime_results = []

for regime in regimes:
    # 전략 수익률
    strat_ret = backtest_by_regime(mom_signal, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크 (같은 레짐 기간)
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(strat_ret) < 3 or len(ew_sub) < 3:
        logger.warning(f"{regime}: 데이터 부족, 스킵")
        continue
    
    strat_perf = calc_perf(strat_ret)
    ew_perf = calc_perf(ew_sub)
    
    # 초과수익 통계
    common_idx = strat_ret.index.intersection(ew_sub.index)
    if len(common_idx) > 0:
        excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": 0}
    
    regime_results.append({
        "Regime": regime,
        "Strat_CAGR": strat_perf["CAGR"] * 100,
        "Strat_Sharpe": strat_perf["Sharpe"],
        "BM_CAGR": ew_perf["CAGR"] * 100,
        "BM_Sharpe": ew_perf["Sharpe"],
        "Excess_CAGR": (strat_perf["CAGR"] - ew_perf["CAGR"]) * 100,
        "t_stat": tstat_info["t_stat"],
        "N_months": len(strat_ret),
    })

regime_df_result = pd.DataFrame(regime_results)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크")
print("="*80)
display(regime_df_result.round(3))

# ─────────────────────────────────────────────────────────────────────────────
# 5) 승패 판정
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*50)
print("레짐별 승패 (Sharpe 기준):")
print("-"*50)

for _, row in regime_df_result.iterrows():
    regime = row["Regime"]
    win = "WIN" if row["Strat_Sharpe"] > row["BM_Sharpe"] else "LOSE"
    excess = row["Excess_CAGR"]
    print(f"{regime:20s} | {win:4s} | 초과수익: {excess:+.1f}%p | t={row['t_stat']:.2f}")

wins = (regime_df_result["Strat_Sharpe"] > regime_df_result["BM_Sharpe"]).sum()
total = len(regime_df_result)
print(f"\n승률: {wins}/{total} ({wins/total*100:.0f}%)")

logger.info("Cell 5 완료: regime_v2 기반 레짐별 검증")

2026-02-06 16:45:25,561 | INFO | regime_m: 770 months
2026-02-06 16:45:25,561 | INFO | 레짐 종류: ['0_Neutral', '2_Recovery_Early', '4_Recovery_Late', '5_Expansion', '6_Peak', '3_Contraction', '1_Crash']
2026-02-06 16:45:25,889 | WARNING | 1_Crash: 데이터 부족, 스킵



레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크


,Regime,Strat_CAGR,Strat_Sharpe,BM_CAGR,BM_Sharpe,Excess_CAGR,t_stat,N_months
0,0_Neutral,35.036,1.870,-15.815,-1.083,50.851,0.863,28
1,2_Recovery_Early,0.536,0.028,42.955,1.762,-42.420,-0.575,3
2,4_Recovery_Late,28.648,1.892,24.031,2.277,4.617,2.435,43
3,5_Expansion,11.886,0.760,40.636,3.791,-28.749,2.243,58
4,6_Peak,-17.757,-1.267,5.011,0.300,-22.768,NaN,5
5,3_Contraction,16.668,0.559,-13.152,-0.470,29.820,11.818,7


2026-02-06 16:45:25,898 | INFO | Cell 5 완료: regime_v2 기반 레짐별 검증



--------------------------------------------------
레짐별 승패 (Sharpe 기준):
--------------------------------------------------
0_Neutral            | WIN  | 초과수익: +50.9%p | t=0.86
2_Recovery_Early     | LOSE | 초과수익: -42.4%p | t=-0.57
4_Recovery_Late      | LOSE | 초과수익: +4.6%p | t=2.43
5_Expansion          | LOSE | 초과수익: -28.7%p | t=2.24
6_Peak               | LOSE | 초과수익: -22.8%p | t=nan
3_Contraction        | WIN  | 초과수익: +29.8%p | t=11.82

승률: 2/6 (33%)


In [7]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 5-1 (결과 해석 + 원인 추정)
# =============================================================================
"""
# Cell 5 결과 해석: 레짐별 모멘텀 성과

## 레짐 타입별 평균 Sharpe

| 레짐 타입 | EW    | MOM_3_1 | MOM_6_0 | MOM_6_1 | MOM_12_0 | MOM_12_1 |
|-----------|-------|---------|---------|---------|----------|----------|
| BULL (n=4)| 2.446 | 1.956   | 1.754   | 1.900   | 2.030    | 1.980    |
| BEAR (n=3)| -1.477| -0.381  | 0.059   | -0.104  | -0.436   | -0.389   |
| SIDE (n=3)| 1.118 | 1.142   | 1.266   | ...     | ...      | ...      |

---

## 충격적인 결과 (직관과 불일치)

### BULL (상승장)
- EW가 모멘텀 전략을 전부 이김 (2.446 vs 최고 2.030)
- 학계 통설: "모멘텀은 추세장에서 유리" -> 불일치

### BEAR (하락장)
- MOM_6_0만 양수 (0.059) - 유일하게 방어
- 롱온리 모멘텀은 하락장에서 기본적으로 털림 (예상대로)

### SIDEWAYS (횡보장)
- MOM_6_0이 EW를 이김 (1.266 vs 1.118)
- 통설: "횡보장에서 모멘텀 안 먹힘" -> 불일치

---

## 원인 후보 (왜 직관과 다른가?)

### 가설 1: 생존편향
- 2013년 기준 "현재" S&P500 종목만 사용
- 상승장에서 살아남은 종목 = 이미 많이 오른 종목
- 모멘텀 상위 선택 = 천장 근처에서 매수

### 가설 2: 상승장 정의 오류 (가장 유력)
- 현재 BULL 정의 = "급락 후 회복기"
- 회복 초반: 낙폭과대주가 더 뜀 (역모멘텀 효과)
- 회복 중후반: 모멘텀 효과 시작
- "급락 후 회복" =/= "완만한 추세 상승"

### 가설 3: EW 벤치마크의 중소형 편향
- S&P500 전종목 동일가중 = 중소형 비중 높음
- 회복기: 중소형이 대형보다 더 급등
- 모멘텀 TOP50 = 대형주 편향 가능성

### 가설 4: Lookback 신호 지연
- 회복 초반: 과거 6~12개월 데이터 = 하락기 데이터
- 급반등 종목을 신호가 늦게 포착
- 포착 시점엔 이미 상승 끝

### 가설 5: TOP_N 설정 문제
- TOP_N = 50 (S&P500의 ~10%)
- 너무 많으면 알파 희석
- 너무 적으면 집중 리스크

### 가설 6: 레짐 기간 임의 설정
- 레짐 시작/종료 시점을 내가 임의로 정함
- 실제 시장 전환점과 불일치 가능성

---

## 검증 우선순위

| 순위 | 가설 | 검증 방법 | 난이도 |
|------|------|----------|--------|
| 1 | 상승장 정의 | 회복 초반 vs 중후반 분리, 완만한 상승기 추가 | 중 |
| 2 | 신호 지연 | Skip 기간 다양화 (3-0, 1-0 등) | 하 |
| 3 | TOP_N | 20, 30, 100 민감도 분석 | 하 |
| 4 | EW 편향 | 시총가중 벤치마크 추가 | 중 |
| 5 | 레짐 기간 | S&P500 지수 차트로 시각적 확인 | 하 |
| 6 | 생존편향 | 검증 불가 (과거 유니버스 없음) | - |

---

## 다음 단계 제안
1. 가설 2 검증: 레짐 재정의 (회복 초반/중반/후반 분리)
2. 가설 3 검증: 시총가중 벤치마크 추가
3. 원인 규명 후 D-3 (변동성 조정 모멘텀) 진행  
어우 두야 ㅋㅋ
"""
print("Cell 5-1: 결과 해석 + 원인 추정 기록 완료")

Cell 5-1: 결과 해석 + 원인 추정 기록 완료


In [8]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 6 (재작성)
# 목적: regime_v2 기반 레짐별 모멘텀 성과 검증
# 산출물: regime_results, sharpe_pivot
# 주의: A팩터와 동일한 레짐/벤치마크 사용
# =============================================================================

TOP_N = 30  # A팩터와 동일

# ─────────────────────────────────────────────────────────────────────────────
# 1) 레짐 데이터 로드
# ─────────────────────────────────────────────────────────────────────────────
regime_df = pd.read_parquet(PATHS["INTERIM"] / "regime_indicators_combined.parquet")

if "date" in regime_df.columns:
    regime_df = regime_df.set_index("date")
regime_df.index = pd.to_datetime(regime_df.index)

# 월말 리샘플링
regime_m = regime_df.resample("ME").last()

logger.info(f"regime_m: {regime_m.shape[0]} months")
logger.info(f"레짐 종류: {regime_m['regime_v2'].unique().tolist()}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 백테스트 함수 (레짐 필터링 버전)
# ─────────────────────────────────────────────────────────────────────────────
def backtest_by_regime(mom_signal: pd.DataFrame, ret_1m: pd.DataFrame,
                       regime_m: pd.DataFrame, regime_col: str, 
                       target_regime: str, top_n: int = 30) -> pd.Series:
    """특정 레짐 기간만 백테스트"""
    port_rets = []
    
    for date in mom_signal.index:
        # 레짐 확인
        if date not in regime_m.index:
            continue
        if regime_m.loc[date, regime_col] != target_regime:
            continue
        if date not in ret_1m.index:
            continue
        
        # 모멘텀 신호
        sig = mom_signal.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        # 상위 N종목
        top_tickers = sig.nlargest(top_n).index
        
        # 다음 달 수익률
        next_idx = ret_1m.index.get_loc(date)
        if next_idx + 1 >= len(ret_1m):
            continue
        next_date = ret_1m.index[next_idx + 1]
        
        # 포트폴리오 수익률
        port_ret = ret_1m.loc[next_date, top_tickers].mean()
        port_rets.append({"date": next_date, "ret": port_ret})
    
    if not port_rets:
        return pd.Series(dtype=float)
    
    return pd.DataFrame(port_rets).set_index("date")["ret"]

# ─────────────────────────────────────────────────────────────────────────────
# 3) 레짐별 백테스트 (MOM_6_0 고정)
# ─────────────────────────────────────────────────────────────────────────────
mom_signal = mom_signals["MOM_6_0"]  # 전체 기간 1위 전략
REGIME_COL = "regime_v2"

regimes = regime_m[REGIME_COL].dropna().unique().tolist()
regime_results = []

for regime in regimes:
    # 전략 수익률
    strat_ret = backtest_by_regime(mom_signal, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크 (같은 레짐 기간)
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(strat_ret) < 3 or len(ew_sub) < 3:
        logger.warning(f"{regime}: 데이터 부족, 스킵")
        continue
    
    # 성과 계산
    strat_perf = calc_perf(strat_ret)
    ew_perf = calc_perf(ew_sub)
    
    # 초과수익 통계
    common_idx = strat_ret.index.intersection(ew_sub.index)
    if len(common_idx) > 0:
        excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": 0}
    
    regime_results.append({
        "Regime": regime,
        "Strat_CAGR": strat_perf["CAGR"] * 100,
        "Strat_Sharpe": strat_perf["Sharpe"],
        "BM_CAGR": ew_perf["CAGR"] * 100,
        "BM_Sharpe": ew_perf["Sharpe"],
        "Excess_CAGR": (strat_perf["CAGR"] - ew_perf["CAGR"]) * 100,
        "t_stat": tstat_info["t_stat"],
        "N_months": len(strat_ret),
    })

regime_df_result = pd.DataFrame(regime_results)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크")
print("="*80)
display(regime_df_result.round(3))

# ─────────────────────────────────────────────────────────────────────────────
# 5) 승패 판정
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*50)
print("레짐별 승패 (Sharpe 기준):")
print("-"*50)

for _, row in regime_df_result.iterrows():
    regime = row["Regime"]
    win = "✓ WIN" if row["Strat_Sharpe"] > row["BM_Sharpe"] else "✗ LOSE"
    excess = row["Excess_CAGR"]
    print(f"{regime:20s} | {win:8s} | 초과수익: {excess:+.1f}%p | t={row['t_stat']:.2f}")

# 승률 계산
wins = (regime_df_result["Strat_Sharpe"] > regime_df_result["BM_Sharpe"]).sum()
total = len(regime_df_result)
print(f"\n승률: {wins}/{total} ({wins/total*100:.0f}%)")

logger.info("Cell 6 완료: regime_v2 기반 레짐별 검증")

2026-02-06 16:45:25,945 | INFO | regime_m: 770 months
2026-02-06 16:45:25,946 | INFO | 레짐 종류: ['0_Neutral', '2_Recovery_Early', '4_Recovery_Late', '5_Expansion', '6_Peak', '3_Contraction', '1_Crash']
2026-02-06 16:45:26,274 | WARNING | 1_Crash: 데이터 부족, 스킵



레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크


,Regime,Strat_CAGR,Strat_Sharpe,BM_CAGR,BM_Sharpe,Excess_CAGR,t_stat,N_months
0,0_Neutral,43.803,2.250,-15.815,-1.083,59.618,1.133,28
1,2_Recovery_Early,7.845,0.378,42.955,1.762,-35.110,-1.750,3
2,4_Recovery_Late,33.198,1.938,24.031,2.277,9.167,2.360,43
3,5_Expansion,13.871,0.828,40.636,3.791,-26.765,2.267,58
4,6_Peak,-26.928,-1.813,5.011,0.300,-31.939,NaN,5
5,3_Contraction,16.867,0.491,-13.152,-0.470,30.019,5.425,7


2026-02-06 16:45:26,284 | INFO | Cell 6 완료: regime_v2 기반 레짐별 검증



--------------------------------------------------
레짐별 승패 (Sharpe 기준):
--------------------------------------------------
0_Neutral            | ✓ WIN    | 초과수익: +59.6%p | t=1.13
2_Recovery_Early     | ✗ LOSE   | 초과수익: -35.1%p | t=-1.75
4_Recovery_Late      | ✗ LOSE   | 초과수익: +9.2%p | t=2.36
5_Expansion          | ✗ LOSE   | 초과수익: -26.8%p | t=2.27
6_Peak               | ✗ LOSE   | 초과수익: -31.9%p | t=nan
3_Contraction        | ✓ WIN    | 초과수익: +30.0%p | t=5.42

승률: 2/6 (33%)


In [9]:
# =============================================================================
# 02_D.ipynb — Cell 6-1 (레짐별 결과 해석)
# =============================================================================
"""
# Cell 5 결과 해석: regime_v2 기반 모멘텀(MOM_6_0) 성과

## 레짐별 성과 요약 (TOP_N=30, MOM_6_0)

| 레짐             | Strat Sharpe | BM Sharpe | 초과수익  | 승패   | N_months |
|------------------|--------------|-----------|-----------|--------|----------|
| 0_Neutral        | 2.252        | -1.083    | +61.6%p   | WIN    | 28       |
| 3_Contraction    | 0.491        | -0.470    | +30.0%p   | WIN    | 7        |
| 4_Recovery_Late  | 1.991        | 2.233     | +10.3%p   | LOSE   | 42       |
| 5_Expansion      | 0.828        | 3.791     | -26.8%p   | LOSE   | 58       |
| 6_Peak           | -1.813       | 0.300     | -31.9%p   | LOSE   | 5        |
| 2_Recovery_Early | 0.378        | 1.762     | -35.1%p   | LOSE   | 3        |
| 1_Crash          | (스킵)       | -         | -         | -      | 0        |

승률: 2/6 (33%)

---

## 핵심 발견

### 1. Neutral (횡보장) — 모멘텀 압승
- 초과수익 +61.6%p, t=1.13
- 방향 없을 때 "오르는 놈 사기"가 유효
- A팩터도 여기서 압승 → 둘 다 비중 부여 가능

### 2. Contraction (하락장) — 모멘텀 방어 성공
- 초과수익 +30.0%p, t=5.42
- 둘 다 Sharpe 낮지만, 모멘텀이 그나마 나음
- "덜 빠지는 놈" 고르는 효과
- A팩터는 여기서 대패 → D만 비중

### 3. Recovery_Early (회복 초반) — 역모멘텀 현상
- 초과수익 -35.1%p, t=-1.75
- 회복 초반 = 낙폭과대주가 급반등
- 모멘텀 상위 = "덜 빠진 놈" = 반등 덜 탐
- 가설 검증됨: Lookback 신호 지연 문제

### 4. Recovery_Late (회복 후반) — BM이 나음
- 초과수익 +10.3%p지만 Sharpe는 BM 승
- 회복 후반 = 시장 전체 상승 → 베타 > 알파
- A팩터는 여기서 승 → A만 비중

### 5. Expansion (확장기) — BM 압승
- BM Sharpe 3.791 vs 전략 0.828
- 확장기엔 그냥 시장 타는 게 나음
- 원인 분석 필요 (셀 5-2 참고)

### 6. Peak (피크) — 모멘텀 털림
- 전략 Sharpe -1.813
- 천장에서 모멘텀 추종 = 꼭대기에서 삼

---

## A팩터 vs D팩터 비교

| 레짐             | A (Value×Catalyst) | D (Momentum) | 권장 비중        |
|------------------|--------------------|--------------|--------------------|
| 0_Neutral        | WIN                | WIN          | 둘 다 비중         |
| 3_Contraction    | LOSE               | WIN          | D만 비중           |
| 4_Recovery_Late  | WIN                | LOSE         | A만 비중           |
| 5_Expansion      | LOSE               | LOSE         | EW (팩터 비중 0)   |
| 6_Peak           | ?                  | LOSE         | 보수적 운용        |
| 2_Recovery_Early | ?                  | LOSE         | 역모멘텀 고려      |

---

## D팩터 잠정 결론

유효 레짐: Neutral, Contraction
무효 레짐: Recovery_Early, Recovery_Late, Expansion, Peak

다음 단계: 유효 레짐 내 Lookback 민감도 테스트
"""
print("Cell 6-1: 레짐별 결과 해석 완료")

Cell 6-1: 레짐별 결과 해석 완료


In [10]:
# =============================================================================
# 02_D.ipynb — Cell 6-2 (Expansion BM 압승 원인 분석)
# =============================================================================
"""
# Expansion(확장기)에서 왜 모멘텀이 EW한테 지는가?

## 현상
- BM Sharpe: 3.791
- 전략 Sharpe: 0.828
- 초과수익: -26.8%p
- 58개월 (표본 충분)

---

## 가설 1: 후발주자 따라잡기 효과

확장기 특징:
- 경기 좋음, 전반적 상승, 낙관론
- "다 오른다"

모멘텀 TOP 30 vs EW 467종목:
- TOP 30: 이미 많이 오른 놈 → 상승 여력 제한
- 나머지: 아직 덜 올랐음 → 따라잡기 시작

확장기엔 선두보다 후발이 더 뛰는 구간 존재
→ 모멘텀 집중 < 전체 분산

검증 방법: 모멘텀 하위 30 vs 상위 30 비교

---

## 가설 2: 중소형 효과

EW 벤치마크 특성:
- S&P500 동일가중 = 중소형 비중 높음
- 시총가중이면 AAPL, MSFT가 7~8%
- 동일가중이면 AAPL도 0.2%, 소형주도 0.2%

확장기 중소형 프리미엄:
- 유동성 풍부 → 위험선호 증가
- 중소형 베팅 활발
- 대형주는 이미 적정가

모멘텀 TOP 30:
- 대형주 편향 가능성 (유동성 높은 놈이 모멘텀 상위)
- 중소형 랠리 수혜 못 받음

검증 방법: 시총가중 벤치마크로 교체 후 비교

---

## 가설 3: 섹터 로테이션

확장기 섹터 순환:
- Q1: 테크 랠리
- Q2: 에너지 랠리
- Q3: 금융 랠리
- ...

모멘텀 신호 = 과거 6개월 기준:
- Q1 끝날 때 테크 상위로 선정
- Q2엔 에너지가 뜨는데 테크 들고 있음
- 로테이션 타이밍 못 맞춤

EW는 전 섹터 보유:
- 자동으로 로테이션 수혜
- 특정 섹터 집중 리스크 없음

검증 방법: 섹터별 수익률 분석, 모멘텀 포트폴리오 섹터 집중도 확인

---

## 검증 우선순위

| 순위 | 가설              | 검증 방법                        | 난이도 |
|------|-------------------|----------------------------------|--------|
| 1    | 중소형 효과       | 시총가중 BM으로 교체             | 중     |
| 2    | 후발 따라잡기     | 모멘텀 하위 30 성과 비교         | 하     |
| 3    | 섹터 로테이션     | 섹터별 수익률, 포트폴리오 집중도 | 상     |

---

## 현재 판단

원인 특정 전에도 결론은 동일:
- Expansion에서 모멘텀 비중 0
- 이유가 뭐든 "안 먹힌다"는 사실은 확정
- 원인 규명은 전략 개선용 (추후 과제)

우선 진행: Lookback 민감도 테스트 (유효 레짐 내)
"""
print("Cell 6-2: Expansion BM 압승 원인 분석 완료")

Cell 6-2: Expansion BM 압승 원인 분석 완료


In [11]:
# =============================================================================
# 02_D.ipynb — Cell 7
# 목적: 레짐별 최적 Lookback 탐색
# 산출물: lookback_results, lookback_pivot
# 주의: 모든 레짐 × 모든 Lookback 조합 테스트
# =============================================================================

from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# 1) 설정
# ─────────────────────────────────────────────────────────────────────────────
LOOKBACKS = ["MOM_3_1", "MOM_6_0", "MOM_6_1", "MOM_12_0", "MOM_12_1"]
REGIME_COL = "regime_v2"
regimes = [r for r in regime_m[REGIME_COL].dropna().unique() if r != "1_Crash"]  # Crash 제외 (데이터 없음)

logger.info(f"Lookback 옵션: {LOOKBACKS}")
logger.info(f"레짐 목록: {regimes}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 전체 조합 백테스트
# ─────────────────────────────────────────────────────────────────────────────
lookback_results = []

total_combinations = len(regimes) * len(LOOKBACKS)
pbar = tqdm(total=total_combinations, desc="Lookback × Regime 테스트")

for regime in regimes:
    # EW 벤치마크 (해당 레짐)
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(ew_sub) < 3:
        pbar.update(len(LOOKBACKS))
        continue
    
    ew_perf = calc_perf(ew_sub)
    
    for lb_name in LOOKBACKS:
        mom_sig = mom_signals[lb_name]
        strat_ret = backtest_by_regime(mom_sig, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
        
        if len(strat_ret) < 3:
            pbar.update(1)
            continue
        
        strat_perf = calc_perf(strat_ret)
        
        # 초과수익 통계
        common_idx = strat_ret.index.intersection(ew_sub.index)
        if len(common_idx) > 0:
            excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
            excess_cagr = strat_perf["CAGR"] - ew_perf["CAGR"]
        else:
            excess_cagr = np.nan
        
        lookback_results.append({
            "Regime": regime,
            "Lookback": lb_name,
            "Strat_Sharpe": strat_perf["Sharpe"],
            "BM_Sharpe": ew_perf["Sharpe"],
            "Excess_CAGR": excess_cagr * 100,
            "N_months": len(strat_ret),
        })
        
        pbar.update(1)

pbar.close()

lookback_df = pd.DataFrame(lookback_results)

# ─────────────────────────────────────────────────────────────────────────────
# 3) 피벗: 레짐 × Lookback Sharpe
# ─────────────────────────────────────────────────────────────────────────────
sharpe_pivot = lookback_df.pivot(index="Regime", columns="Lookback", values="Strat_Sharpe")
sharpe_pivot = sharpe_pivot[LOOKBACKS].round(3)  # 컬럼 순서 정리

print("\n" + "="*80)
print("레짐 × Lookback: 전략 Sharpe")
print("="*80)
display(sharpe_pivot)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 피벗: 레짐 × Lookback 초과수익
# ─────────────────────────────────────────────────────────────────────────────
excess_pivot = lookback_df.pivot(index="Regime", columns="Lookback", values="Excess_CAGR")
excess_pivot = excess_pivot[LOOKBACKS].round(1)

print("\n" + "="*80)
print("레짐 × Lookback: 초과수익 (%p)")
print("="*80)
display(excess_pivot)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 레짐별 최적 Lookback 추출
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*60)
print("레짐별 최적 Lookback (Sharpe 기준):")
print("-"*60)

best_lookback = {}
for regime in sharpe_pivot.index:
    row = sharpe_pivot.loc[regime]
    best_lb = row.idxmax()
    best_sharpe = row.max()
    
    # 해당 레짐 BM Sharpe
    bm_sharpe = lookback_df[lookback_df["Regime"] == regime]["BM_Sharpe"].iloc[0]
    
    win = "WIN" if best_sharpe > bm_sharpe else "LOSE"
    diff = best_sharpe - bm_sharpe
    
    best_lookback[regime] = best_lb
    print(f"{regime:20s} -> {best_lb:10s} (Sharpe {best_sharpe:.3f}) | BM {bm_sharpe:.3f} | {win} ({diff:+.3f})")

# ─────────────────────────────────────────────────────────────────────────────
# 6) 최적 Lookback으로도 BM 못 이기는 레짐 표시
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*60)
print("결론: 레짐별 모멘텀 사용 권장")
print("-"*60)

for regime in sharpe_pivot.index:
    row = sharpe_pivot.loc[regime]
    best_lb = row.idxmax()
    best_sharpe = row.max()
    bm_sharpe = lookback_df[lookback_df["Regime"] == regime]["BM_Sharpe"].iloc[0]
    
    if best_sharpe > bm_sharpe:
        print(f"{regime:20s} -> 사용 O | 최적: {best_lb}")
    else:
        print(f"{regime:20s} -> 사용 X | (최적 {best_lb}로도 BM한테 짐)")

logger.info("Cell 7 완료: 레짐별 Lookback 민감도 테스트")

2026-02-06 16:45:26,329 | INFO | Lookback 옵션: ['MOM_3_1', 'MOM_6_0', 'MOM_6_1', 'MOM_12_0', 'MOM_12_1']
2026-02-06 16:45:26,330 | INFO | 레짐 목록: ['0_Neutral', '2_Recovery_Early', '4_Recovery_Late', '5_Expansion', '6_Peak', '3_Contraction']
Lookback × Regime 테스트: 100%|██████████| 30/30 [00:01<00:00, 20.15it/s]


레짐 × Lookback: 전략 Sharpe


Lookback,MOM_3_1,MOM_6_0,MOM_6_1,MOM_12_0,MOM_12_1
Regime,,,,,
0_Neutral,1.421,2.250,1.365,1.852,1.292
2_Recovery_Early,0.289,0.378,0.313,-1.577,-1.342
3_Contraction,0.799,0.491,0.638,0.667,0.896
4_Recovery_Late,1.322,1.938,1.734,1.529,1.281
5_Expansion,1.059,0.828,0.954,0.915,0.889
6_Peak,-0.625,-1.813,-2.379,-0.743,-0.927



레짐 × Lookback: 초과수익 (%p)


Lookback,MOM_3_1,MOM_6_0,MOM_6_1,MOM_12_0,MOM_12_1
Regime,,,,,
0_Neutral,43.6,59.6,49.7,55.9,46.6
2_Recovery_Early,-35.2,-35.1,-35.8,-64.2,-64.1
3_Contraction,36.7,30.0,36.0,40.0,49.6
4_Recovery_Late,2.1,9.2,5.4,2.4,-0.2
5_Expansion,-23.0,-26.8,-25.3,-26.7,-27.9
6_Peak,NaN,NaN,NaN,NaN,NaN


2026-02-06 16:45:27,856 | INFO | Cell 7 완료: 레짐별 Lookback 민감도 테스트



------------------------------------------------------------
레짐별 최적 Lookback (Sharpe 기준):
------------------------------------------------------------
0_Neutral            -> MOM_6_0    (Sharpe 2.250) | BM -1.083 | WIN (+3.333)
2_Recovery_Early     -> MOM_6_0    (Sharpe 0.378) | BM 1.762 | LOSE (-1.384)
3_Contraction        -> MOM_12_1   (Sharpe 0.896) | BM -0.470 | WIN (+1.366)
4_Recovery_Late      -> MOM_6_0    (Sharpe 1.938) | BM 2.277 | LOSE (-0.339)
5_Expansion          -> MOM_3_1    (Sharpe 1.059) | BM 3.791 | LOSE (-2.732)
6_Peak               -> MOM_3_1    (Sharpe -0.625) | BM 0.300 | LOSE (-0.925)

------------------------------------------------------------
결론: 레짐별 모멘텀 사용 권장
------------------------------------------------------------
0_Neutral            -> 사용 O | 최적: MOM_6_0
2_Recovery_Early     -> 사용 X | (최적 MOM_6_0로도 BM한테 짐)
3_Contraction        -> 사용 O | 최적: MOM_12_1
4_Recovery_Late      -> 사용 X | (최적 MOM_6_0로도 BM한테 짐)
5_Expansion          -> 사용 X | (최적 MOM_3_1로도 BM한테 짐

In [12]:
# =============================================================================
# 02_D.ipynb — Cell 7-1 (Lookback 민감도 결과 해석)
# =============================================================================
"""
# Cell 7 결과 해석: 레짐별 최적 Lookback

## 비교 구조 설명

전략 (MOM_X_Y):
  - 해당 레짐 기간 동안
  - 모멘텀 상위 30종목 동일가중
  - Lookback 방식에 따라 종목 선정 다름

벤치마크 (EW):
  - 해당 레짐 기간 동안
  - 전종목 467개 동일가중
  - Lookback과 무관하게 레짐별 고정값

---

## 레짐 × Lookback: 전략 Sharpe

| Regime           | MOM_3_1 | MOM_6_0 | MOM_6_1 | MOM_12_0 | MOM_12_1 |
|------------------|---------|---------|---------|----------|----------|
| 0_Neutral        | 1.460   | 2.252   | 1.402   | 1.863    | 1.325    |
| 2_Recovery_Early | 0.289   | 0.378   | 0.313   | -1.577   | -1.342   |
| 3_Contraction    | 0.799   | 0.491   | 0.638   | 0.667    | 0.896    |
| 4_Recovery_Late  | 1.348   | 1.991   | 1.767   | 1.592    | 1.345    |
| 5_Expansion      | 1.059   | 0.828   | 0.954   | 0.915    | 0.889    |
| 6_Peak           | -0.625  | -1.813  | -2.379  | -0.743   | -0.927   |

---

## 레짐 × Lookback: 초과수익 (%p)

| Regime           | MOM_3_1 | MOM_6_0 | MOM_6_1 | MOM_12_0 | MOM_12_1 |
|------------------|---------|---------|---------|----------|----------|
| 0_Neutral        | 44.8    | 61.6    | 51.5    | 58.0     | 48.6     |
| 2_Recovery_Early | -35.2   | -35.1   | -35.8   | -64.2    | -64.1    |
| 3_Contraction    | 36.7    | 30.0    | 36.0    | 40.0     | 49.6     |
| 4_Recovery_Late  | 2.9     | 10.3    | 6.3     | 3.7      | 1.1      |
| 5_Expansion      | -23.0   | -26.8   | -25.3   | -26.7    | -27.9    |
| 6_Peak           | NaN     | NaN     | NaN     | NaN      | NaN      |

---

## 레짐별 최적 Lookback + 승패

| 레짐             | 최적 Lookback | Sharpe | BM Sharpe | 승패           |
|------------------|---------------|--------|-----------|----------------|
| 0_Neutral        | MOM_6_0       | 2.252  | -1.083    | WIN (+3.335)   |
| 2_Recovery_Early | MOM_6_0       | 0.378  | 1.762     | LOSE (-1.384)  |
| 3_Contraction    | MOM_12_1      | 0.896  | -0.470    | WIN (+1.366)   |
| 4_Recovery_Late  | MOM_6_0       | 1.991  | 2.233     | LOSE (-0.242)  |
| 5_Expansion      | MOM_3_1       | 1.059  | 3.791     | LOSE (-2.732)  |
| 6_Peak           | MOM_3_1       | -0.625 | 0.300     | LOSE (-0.925)  |

---

## 핵심 발견

### 1. 유효 레짐 확정
- Neutral: WIN (어떤 Lookback이든 BM 압도)
- Contraction: WIN (MOM_12_1이 최적)
- 나머지: LOSE (어떤 Lookback으로도 BM한테 짐)

### 2. 최적 Lookback이 레짐마다 다름

| 유효 레짐     | 최적 Lookback | 이유                              |
|---------------|---------------|-----------------------------------|
| 0_Neutral     | MOM_6_0       | 횡보장, 단기 추세가 유효          |
| 3_Contraction | MOM_12_1      | 하락장, 장기+노이즈 제거가 유효   |

### 3. Neutral에서 MOM_6_0 > MOM_12_0 이유
- 횡보장 = 방향 없음, 단기 오르내림
- 12개월은 너무 오래된 정보 포함
- 6개월이 "최근 모멘텀"을 더 잘 포착

### 4. Contraction에서 MOM_12_1 > MOM_6_0 이유
- 하락장 = 변동성 높음, 노이즈 심함
- 단기(6개월)는 노이즈에 휘둘림
- 장기(12개월) + skip=1 → 안정적 추세 + 급락 노이즈 회피

---

## D팩터 최종 설정 (잠정)

| 레짐          | 사용 여부 | Lookback  | 초과수익 |
|---------------|-----------|-----------|----------|
| 0_Neutral     | O         | MOM_6_0   | +61.6%p  |
| 3_Contraction | O         | MOM_12_1  | +49.6%p  |
| 그 외         | X         | -         | -        |

---

## 다음 단계
1. D-3 (변동성 조정 모멘텀) 검토 — 추가 개선 가능성
2. D-4 (하방 방어형 모멘텀) 검토
3. 또는 현재 결과로 D팩터 확정 후 멀티팩터 조합으로 이동
!! 참고로 D-2는 A와 상관관계로 문제 발생 아래 참고
 1. A에서 모멘텀 필터 추가하는 방법, 
 2. 그냥 D-2제끼는 방법, 
 3. 상관관계 테스트후 낮을 때만 사용??
"""
print("Cell 7-1: Lookback 민감도 결과 해석 완료")

Cell 7-1: Lookback 민감도 결과 해석 완료


In [13]:
# =============================================================================
# 02_D.ipynb — Cell 8
# 목적: D-3 변동성 조정 모멘텀 (ret / vol)
# 산출물: mom_vol_adjusted, 레짐별 성과 비교
# 주의: 동일 수익률이면 변동성 낮은 놈이 상위
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 변동성 계산 (과거 N개월 수익률의 표준편차)
# ─────────────────────────────────────────────────────────────────────────────
def calc_volatility(ret_1m: pd.DataFrame, lookback: int = 6) -> pd.DataFrame:
    """월간 수익률의 rolling 표준편차 (연환산)"""
    vol = ret_1m.rolling(window=lookback, min_periods=3).std() * np.sqrt(12)
    return vol

# 6개월 변동성
vol_6m = calc_volatility(ret_1m, lookback=6)

logger.info(f"vol_6m: {vol_6m.shape}")
logger.info(f"샘플 (최근 3개월, 5종목):\n{vol_6m.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 변동성 조정 모멘텀: MOM / VOL
# ─────────────────────────────────────────────────────────────────────────────
# 기존 MOM_6_0 사용
mom_raw = mom_signals["MOM_6_0"]

# 변동성 조정 (Sharpe-like ratio)
mom_vol_adj = mom_raw / vol_6m

# inf, -inf 처리
mom_vol_adj = mom_vol_adj.replace([np.inf, -np.inf], np.nan)

logger.info(f"mom_vol_adj 샘플:\n{mom_vol_adj.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 레짐별 백테스트: MOM_6_0 vs MOM_6_0_VOL_ADJ
# ─────────────────────────────────────────────────────────────────────────────
REGIME_COL = "regime_v2"
regimes = [r for r in regime_m[REGIME_COL].dropna().unique() if r != "1_Crash"]

comparison_results = []

for regime in regimes:
    # 기존 MOM_6_0
    ret_raw = backtest_by_regime(mom_raw, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # 변동성 조정 MOM
    ret_vol = backtest_by_regime(mom_vol_adj, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(ret_raw) < 3 or len(ret_vol) < 3:
        continue
    
    perf_raw = calc_perf(ret_raw)
    perf_vol = calc_perf(ret_vol)
    perf_ew = calc_perf(ew_sub)
    
    comparison_results.append({
        "Regime": regime,
        "MOM_6_0_Sharpe": perf_raw["Sharpe"],
        "MOM_VOL_ADJ_Sharpe": perf_vol["Sharpe"],
        "BM_Sharpe": perf_ew["Sharpe"],
        "VOL_ADJ_vs_RAW": perf_vol["Sharpe"] - perf_raw["Sharpe"],
        "N_months": len(ret_raw),
    })

comp_df = pd.DataFrame(comparison_results).round(3)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("D-3: 변동성 조정 모멘텀 vs 기존 모멘텀")
print("="*80)
display(comp_df)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 개선 여부 판정
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*50)
print("변동성 조정 효과 (VOL_ADJ - RAW):")
print("-"*50)

for _, row in comp_df.iterrows():
    regime = row["Regime"]
    diff = row["VOL_ADJ_vs_RAW"]
    result = "개선" if diff > 0 else "악화"
    print(f"{regime:20s} | {result:4s} ({diff:+.3f})")

improved = (comp_df["VOL_ADJ_vs_RAW"] > 0).sum()
total = len(comp_df)
print(f"\n개선 레짐: {improved}/{total}")

logger.info("Cell 8 완료: D-3 변동성 조정 모멘텀 테스트")

2026-02-06 16:45:27,907 | INFO | vol_6m: (153, 467)
2026-02-06 16:45:27,910 | INFO | 샘플 (최근 3개월, 5종목):
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.311  0.187  0.213  0.187  0.203
2026-01-31  0.307  0.228  0.227  0.237  0.173
2026-02-28  0.295  0.192  0.188  0.209  0.172
2026-02-06 16:45:27,914 | INFO | mom_vol_adj 샘플:
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.507  1.751  1.177 -0.377  0.264
2026-01-31  0.554  1.108  0.871 -0.530  0.671
2026-02-28  0.121  0.989  0.304 -0.814  0.665



D-3: 변동성 조정 모멘텀 vs 기존 모멘텀


,Regime,MOM_6_0_Sharpe,MOM_VOL_ADJ_Sharpe,BM_Sharpe,VOL_ADJ_vs_RAW,N_months
0,0_Neutral,2.250,2.473,-1.083,0.223,28
1,2_Recovery_Early,0.378,-0.945,1.762,-1.323,3
2,4_Recovery_Late,1.938,1.361,2.277,-0.577,43
3,5_Expansion,0.828,0.995,3.791,0.167,58
4,6_Peak,-1.813,-0.893,0.300,0.920,5
5,3_Contraction,0.491,-0.018,-0.470,-0.509,7


2026-02-06 16:45:28,522 | INFO | Cell 8 완료: D-3 변동성 조정 모멘텀 테스트



--------------------------------------------------
변동성 조정 효과 (VOL_ADJ - RAW):
--------------------------------------------------
0_Neutral            | 개선   (+0.223)
2_Recovery_Early     | 악화   (-1.323)
4_Recovery_Late      | 악화   (-0.577)
5_Expansion          | 개선   (+0.167)
6_Peak               | 개선   (+0.920)
3_Contraction        | 악화   (-0.509)

개선 레짐: 3/6


In [14]:

""""
## 변동성 조정 논리
    수익    변동성  vol_adj
A   30%     15%     2.000
B   30%     30%     1.000
C   20%     10%     2.000
즉 C가 변동성 조정되면 Shape 값이 B보다 커짐  수익은 딸리지만 C 선택

## 핵심 발견

### 1. Neutral에서 개선 (2.252 -> 2.471)
- 원래 유효했던 레짐에서 추가 개선
- 횡보장에서 "진짜 추세" vs "노이즈 상승" 구별 효과

### 2. Peak에서 방어력 개선 (-1.813 -> -0.893)
- 여전히 음수지만 덜 털림
- 피크에서 변동성 낮은 종목이 급락 방어

### 3. Contraction에서 악화 (0.491 -> -0.018)
- D-1에서 유효했던 레짐이 D-3에서 망함
- 하락장에서는 변동성 조정이 오히려 독
- 원인: 하락장에서 "변동성 낮음" = "덜 빠짐" = "반등도 덜 함"

### 4. Recovery 계열 전부 악화
- 회복기에는 출렁이면서 급등하는 종목이 더 뜀
- 변동성 낮은 안정주는 회복 랠리 수혜 못 받음

---

## D-3 가설 검증 결과

가설: 변동성 낮은 상승이 더 지속된다
결과: 부분적으로 맞음

- Neutral/Peak: 맞음 (노이즈 적은 추세가 유리)
- Contraction/Recovery: 틀림 (변동성 높은 놈이 더 뜀)

---

## D-1 vs D-3 레짐별 선택

| 레짐          | D-1 (MOM_6_0) | D-3 (VOL_ADJ) | 선택   |
|---------------|---------------|---------------|--------|
| 0_Neutral     | WIN           | 더 WIN        | D-3    |
| 3_Contraction | WIN           | LOSE          | D-1    |
| 나머지        | LOSE          | LOSE          | 안 씀  |

---

## 실전 적용

Neutral 레짐:
- D-3 (변동성 조정 모멘텀) 사용
- MOM_6_0 / VOL_6M 기준 상위 30종목

Contraction 레짐:
- D-1 (기존 모멘텀) 사용
- MOM_12_1 기준 상위 30종목 (Cell 7 결과)

기타 레짐:
- 모멘텀 비중 0
"""
print("Cell 8-1: D-3 결과 해석 완료")

Cell 8-1: D-3 결과 해석 완료


In [15]:
# =============================================================================
# 02_D.ipynb — Cell 9
# 목적: D-4 하방 방어형 모멘텀 (조정 시 덜 빠지는 놈)
# 산출물: mom_downside_adj, 레짐별 성과 비교
# 주의: 상승률 대비 하락 방어력 반영
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 하방 변동성 계산 (Downside Deviation)
# ─────────────────────────────────────────────────────────────────────────────
def calc_downside_vol(ret_1m: pd.DataFrame, lookback: int = 6) -> pd.DataFrame:
    """
    하방 변동성: 음수 수익률만으로 계산한 표준편차
    - 상승은 무시, 하락만 봄
    - 조정 시 덜 빠지는 종목 = 하방 변동성 낮음
    """
    # 음수만 남기고 나머지는 0
    neg_ret = ret_1m.clip(upper=0)
    
    # rolling std (연환산)
    downside_vol = neg_ret.rolling(window=lookback, min_periods=3).std() * np.sqrt(12)
    
    return downside_vol

# 6개월 하방 변동성
downside_vol_6m = calc_downside_vol(ret_1m, lookback=6)

logger.info(f"downside_vol_6m: {downside_vol_6m.shape}")
logger.info(f"샘플 (최근 3개월, 5종목):\n{downside_vol_6m.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 일반 변동성 vs 하방 변동성 비교
# ─────────────────────────────────────────────────────────────────────────────
print("\n변동성 비교 (최근 시점, 5종목):")
print("-" * 50)
print(f"전체 변동성:\n{vol_6m.iloc[-1, :5].round(3)}")
print(f"하방 변동성:\n{downside_vol_6m.iloc[-1, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 하방 조정 모멘텀: MOM / Downside Vol
# ─────────────────────────────────────────────────────────────────────────────
mom_raw = mom_signals["MOM_6_0"]

# 하방 변동성 조정 (Sortino-like ratio)
mom_downside_adj = mom_raw / downside_vol_6m

# inf, -inf 처리
mom_downside_adj = mom_downside_adj.replace([np.inf, -np.inf], np.nan)

logger.info(f"mom_downside_adj 샘플:\n{mom_downside_adj.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 레짐별 백테스트: MOM_6_0 vs D-3 vs D-4
# ─────────────────────────────────────────────────────────────────────────────
REGIME_COL = "regime_v2"
regimes = [r for r in regime_m[REGIME_COL].dropna().unique() if r != "1_Crash"]

comparison_results = []

for regime in regimes:
    # 기존 MOM_6_0 (D-1)
    ret_raw = backtest_by_regime(mom_raw, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # 변동성 조정 (D-3)
    ret_vol = backtest_by_regime(mom_vol_adj, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # 하방 조정 (D-4)
    ret_down = backtest_by_regime(mom_downside_adj, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(ret_raw) < 3 or len(ret_down) < 3:
        continue
    
    perf_raw = calc_perf(ret_raw)
    perf_vol = calc_perf(ret_vol)
    perf_down = calc_perf(ret_down)
    perf_ew = calc_perf(ew_sub)
    
    comparison_results.append({
        "Regime": regime,
        "D1_MOM_6_0": perf_raw["Sharpe"],
        "D3_VOL_ADJ": perf_vol["Sharpe"],
        "D4_DOWN_ADJ": perf_down["Sharpe"],
        "BM_Sharpe": perf_ew["Sharpe"],
        "Best": max(perf_raw["Sharpe"], perf_vol["Sharpe"], perf_down["Sharpe"]),
        "N_months": len(ret_raw),
    })

comp_df = pd.DataFrame(comparison_results).round(3)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("D-1 vs D-3 vs D-4 비교 (Sharpe)")
print("="*80)
display(comp_df)

# ─────────────────────────────────────────────────────────────────────────────
# 6) 레짐별 최적 전략
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*60)
print("레짐별 최적 모멘텀 전략:")
print("-"*60)

for _, row in comp_df.iterrows():
    regime = row["Regime"]
    d1 = row["D1_MOM_6_0"]
    d3 = row["D3_VOL_ADJ"]
    d4 = row["D4_DOWN_ADJ"]
    bm = row["BM_Sharpe"]
    
    # 최고 전략 찾기
    best_val = max(d1, d3, d4)
    if best_val == d1:
        best_name = "D-1"
    elif best_val == d3:
        best_name = "D-3"
    else:
        best_name = "D-4"
    
    # BM 대비 승패
    win = "WIN" if best_val > bm else "LOSE"
    
    print(f"{regime:20s} | 최적: {best_name} ({best_val:.3f}) | BM: {bm:.3f} | {win}")

logger.info("Cell 9 완료: D-4 하방 방어형 모멘텀 테스트")

2026-02-06 16:45:28,582 | INFO | downside_vol_6m: (153, 467)
2026-02-06 16:45:28,585 | INFO | 샘플 (최근 3개월, 5종목):
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.157  0.035  0.073  0.120  0.090
2026-01-31  0.158  0.067  0.072  0.176  0.068
2026-02-28  0.154  0.067  0.070  0.175  0.068
2026-02-06 16:45:28,590 | INFO | mom_downside_adj 샘플:
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  1.003  9.248  3.439 -0.589  0.596
2026-01-31  1.079  3.771  2.754 -0.713  1.718
2026-02-28  0.231  2.831  0.818 -0.971  1.691



변동성 비교 (최근 시점, 5종목):
--------------------------------------------------
전체 변동성:
ticker
A       0.295
AAPL    0.192
ABBV    0.188
ABT     0.209
ACGL    0.172
Name: 2026-02-28 00:00:00, dtype: float64
하방 변동성:
ticker
A       0.154
AAPL    0.067
ABBV    0.070
ABT     0.175
ACGL    0.068
Name: 2026-02-28 00:00:00, dtype: float64

D-1 vs D-3 vs D-4 비교 (Sharpe)


,Regime,D1_MOM_6_0,D3_VOL_ADJ,D4_DOWN_ADJ,BM_Sharpe,Best,N_months
0,0_Neutral,2.250,2.473,2.316,-1.083,2.473,28
1,2_Recovery_Early,0.378,-0.945,-0.373,1.762,0.378,3
2,4_Recovery_Late,1.938,1.361,1.096,2.277,1.938,43
3,5_Expansion,0.828,0.995,0.872,3.791,0.995,58
4,6_Peak,-1.813,-0.893,-0.726,0.300,-0.726,5
5,3_Contraction,0.491,-0.018,0.229,-0.470,0.491,7


2026-02-06 16:45:29,445 | INFO | Cell 9 완료: D-4 하방 방어형 모멘텀 테스트



------------------------------------------------------------
레짐별 최적 모멘텀 전략:
------------------------------------------------------------
0_Neutral            | 최적: D-3 (2.473) | BM: -1.083 | WIN
2_Recovery_Early     | 최적: D-1 (0.378) | BM: 1.762 | LOSE
4_Recovery_Late      | 최적: D-1 (1.938) | BM: 2.277 | LOSE
5_Expansion          | 최적: D-3 (0.995) | BM: 3.791 | LOSE
6_Peak               | 최적: D-4 (-0.726) | BM: 0.300 | LOSE
3_Contraction        | 최적: D-1 (0.491) | BM: -0.470 | WIN


In [16]:
# =============================================================================
# 02_D.ipynb — Cell 9-1 (D-4 결과 해석)
# =============================================================================
"""
# D-4 하방 방어형 모멘텀 결과

## D-1 vs D-3 vs D-4 비교 (Sharpe)

| 레짐             | D-1   | D-3   | D-4   | BM     | 최적 | 승패 |
|------------------|-------|-------|-------|--------|------|------|
| 0_Neutral        | 2.252 | 2.471 | 2.342 | -1.083 | D-3  | WIN  |
| 3_Contraction    | 0.491 | -0.018| 0.229 | -0.470 | D-1  | WIN  |
| 4_Recovery_Late  | 1.991 | 1.369 | 1.129 | 2.233  | D-1  | LOSE |
| 5_Expansion      | 0.828 | 0.995 | 0.872 | 3.791  | D-3  | LOSE |
| 6_Peak           | -1.813| -0.893| -0.726| 0.300  | D-4  | LOSE |
| 2_Recovery_Early | 0.378 | -0.945| -0.373| 1.762  | D-1  | LOSE |

## D-4 버리는 이유

### 1. 유효 레짐에서 1등 못 함
- Neutral: D-3가 1등
- Contraction: D-1이 1등
- D-4가 1등인 Peak은 어차피 LOSE 레짐

### 2. D-3 vs D-4 차이
- D-3 (전체 변동성): 상승+하락 모두 고려 -> 정보량 많음
- D-4 (하방 변동성): 하락만 고려 -> 정보 손실

### 3. 하방 변동성의 실무 용도
- 종목 선정 시그널: 효과 제한적 (우리가 테스트한 것)
- 포트폴리오 평가: Sortino Ratio로 사용 (실무)
- 숏 타겟 선정: 하방 변동성 높은 놈 숏 (롱숏 전략)
- 테일 리스크 필터: 극단적 손실 회피 (기관)

### 4. 결론
- 우리 용도 (롱온리, 종목 선정)에서는 D-4 무의미
- D-3가 상위호환
- D-4 폐기
"""
print("Cell 9-1: D-4 결과 해석 완료")

Cell 9-1: D-4 결과 해석 완료


In [17]:
# =============================================================================
# 02_D.ipynb — Cell 10 (D팩터 최종 결론)
# =============================================================================
"""
# D팩터 (모멘텀) 최종 결론

## 테스트 요약

| 가설 | 내용 | 결과 |
|------|------|------|
| D-1  | 추세 지속 (단순 모멘텀) | 조건부 유효 |
| D-2  | 실적 기반 모멘텀 | 생략 (A팩터와 상관 문제) |
| D-3  | 변동성 조정 모멘텀 | 조건부 유효 (Neutral 한정) |
| D-4  | 하방 방어형 모멘텀 | 폐기 |

---

## 레짐별 최적 전략

| 레짐          | 사용 | 전략 | Lookback | Sharpe |
|---------------|------|------|----------|--------|
| 0_Neutral     | O    | D-3  | MOM_6_0 / VOL | 2.471 |
| 3_Contraction | O    | D-1  | MOM_12_1 | 0.896 |
| 4_Recovery_Late | X  | -    | - | BM한테 짐 |
| 5_Expansion   | X    | -    | - | BM한테 짐 |
| 6_Peak        | X    | -    | - | BM한테 짐 |
| 2_Recovery_Early | X | -   | - | BM한테 짐 |

---

## A팩터 vs D팩터 비교

| 레짐          | A (Value×Catalyst) | D (Momentum) | 권장       |
|---------------|--------------------|--------------|--------------------|
| 0_Neutral     | WIN                | WIN (D-3)    | 둘 다 비중         |
| 3_Contraction | LOSE               | WIN (D-1)    | D만 비중           |
| 4_Recovery_Late | WIN              | LOSE         | A만 비중           |
| 5_Expansion   | LOSE               | LOSE         | EW (팩터 비중 0)   |

---

## 실전 적용 요약

Neutral (횡보장):
- A팩터: 사용 O (LAG=45일)
- D팩터: 사용 O (D-3, MOM_6_0/VOL, TOP_N=30)

Contraction (하락장):
- A팩터: 사용 X
- D팩터: 사용 O (D-1, MOM_12_1, TOP_N=30)

Recovery_Late (회복 후반):
- A팩터: 사용 O (LAG=30일)
- D팩터: 사용 X

Expansion (확장기):
- A팩터: 사용 X
- D팩터: 사용 X
- 그냥 EW 또는 시장 베타 타기

---

## 핵심 인사이트

1. 모멘텀은 "추세장에서 유리하다"는 통설과 다름
   - 확장기(상승 추세): BM한테 짐
   - 횡보장: 오히려 압승
   - 하락장: 방어 효과 있음

2. Lookback은 레짐마다 달라야 함
   - Neutral: 단기 (6개월, skip 없음)
   - Contraction: 장기 (12개월, skip 1개월)

3. 변동성 조정은 횡보장에서만 유효
   - Neutral: D-3 > D-1
   - Contraction: D-1 > D-3

4. A팩터와 상호보완적
   - 유효 레짐이 다름
   - 멀티팩터 조합 시 분산 효과 기대

---

## 다음 단계

1. 멀티팩터 조합 (A + D)
2. 레짐 판단 로직과 연결
3. 백테스트 (레짐 스위칭 전략)
"""
print("Cell 10: D팩터 최종 결론 완료")

Cell 10: D팩터 최종 결론 완료
